# CIFAR-10 split and class-distribution check

This notebook checks what data the local experiment is actually evaluating: split sizes and class balance for train, retain, forget, validation, and test. It does not train models or compute the official unlearning score. In the CASIA-SURF competition, the labels were age groups and forget data was subject-separated; CIFAR-10 is a local proxy with object classes and random challenge forget indices.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
from torch.utils.data import Subset


'''repo_root = Path.cwd()
if not (repo_root / "src" / "unlearn").is_dir():
    raise RuntimeError("Run this notebook with the repository root as the working directory.")
sys.path.insert(0, str(repo_root / "src")) '''

from unlearn.data import get_cifar10_loaders

loaders = get_cifar10_loaders(
    root=str(repo_root / "Notebooks" / "data"),
    forget_idx_path=str(repo_root / "Notebooks" / "forget_idx.npy"),
    batch_size=128,
    num_workers=0,
    seed=42,
)
class_names = loaders["train"].dataset.classes
print("Classes:", class_names)

RuntimeError: Run this notebook with the repository root as the working directory.

In [ ]:
def labels_for(dataset):
    if isinstance(dataset, Subset):
        parent_labels = labels_for(dataset.dataset)
        return parent_labels[np.asarray(dataset.indices, dtype=np.int64)]
    if not hasattr(dataset, "targets"):
        raise TypeError(f"Cannot read labels from {type(dataset).__name__}.")
    return np.asarray(dataset.targets, dtype=np.int64)

split_labels = {
    name: labels_for(loader.dataset)
    for name, loader in loaders.items()
}
counts = {
    name: np.bincount(labels, minlength=len(class_names))
    for name, labels in split_labels.items()
}

print(f"{'split':<10} {'size':>8}  class counts in CIFAR-10 order")
for name, class_counts in counts.items():
    print(f"{name:<10} {class_counts.sum():>8}  {class_counts.tolist()}")

print("\nCheck: retain + forget = train:",
      len(loaders['retain'].dataset) + len(loaders['forget'].dataset)
      == len(loaders['train'].dataset))

In [ ]:
split_names = ["retain", "forget", "val", "test"]
fig, ax = plt.subplots(figsize=(11, 5))
x = np.arange(len(class_names))
width = 0.8 / len(split_names)
for index, name in enumerate(split_names):
    total = counts[name].sum()
    percentages = 100 * counts[name] / total if total else np.zeros(len(class_names))
    ax.bar(x + index * width, percentages, width, label=f"{name} (n={total})")
ax.set_xticks(x + width * (len(split_names) - 1) / 2, class_names, rotation=30, ha="right")
ax.set_ylabel("Examples per class (%)")
ax.set_title("CIFAR-10 class mix by split")
ax.legend()
fig.tight_layout()
plt.show()

## How to read this

`forget` and `retain` partition the training data; `val` and `test` are separate halves of the CIFAR-10 held-out set. Compare the forget bars with the retain bars to see whether the random forget-index sample has class imbalance. This is data context only: it is not an unlearning result, and these CIFAR labels are not the CASIA age-group labels.